##**DINO-Style-Self-Supervised-Learning-from-Scratch**

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

Cuda

In [ ]:
if torch.cuda.is_available:
  device = 'cuda'
else:
  device = 'cpu'
print(f"device: {device}")

ViT

In [ ]:
class PatchEmbedding(nn.Module):
  def __init__(self, img_channel, patch_size, d_model):
    super().__init__()

    self.Conv = nn.Conv2d(img_channel, d_model, patch_size, patch_size, bias=False)

  def forward(self, x):
    x = self.Conv(x)

    x = x.flatten(2).transpose(1,2)

    return x

In [ ]:
import math

class Attention(nn.Module):
  def __init__(self, d_model, num_heads, save_attention=False):
    super().__init__()

    d_k = d_model // num_heads

    self.d_model = d_model
    self.d_k = d_k
    self.num_heads = num_heads

    self.W_Q = nn.Linear(d_model,d_model)
    self.W_K = nn.Linear(d_model,d_model)
    self.W_V = nn.Linear(d_model,d_model)
    self.W_O = nn.Linear(d_model,d_model)

    self.save_attention = save_attention
    self.attention_map = None

  def forward(self,q,k,v,mask=None):
    batch_size = q.size(0)

    # q,k,v -> Q,K,V
    Q = self.W_Q(q)
    K = self.W_K(k)
    V = self.W_V(v)

    # Multi head
    Q = Q.reshape(batch_size,-1,self.num_heads,self.d_k).transpose(1,2)
    K = K.reshape(batch_size,-1,self.num_heads,self.d_k).transpose(1,2)
    V = V.reshape(batch_size,-1,self.num_heads,self.d_k).transpose(1,2)

    # Attention
    attn_score = torch.matmul(Q, K.transpose(-1,-2)) / math.sqrt(self.d_k)
    if mask is not None:
      attn_score = torch.masked_fill(attn_score, mask==0, -1e9) # mask = [1, 1, 1, ... 0, 0, 0]
    attn_score = torch.softmax(attn_score, dim=-1)
    out = torch.matmul(attn_score, V)

    if self.save_attention:
        self.attention_map = attn_score #(B,heads,N+1,N+1)

    # Concatenate
    out = out.transpose(1,2).reshape(batch_size,-1,self.d_model)
    out = self.W_O(out)

    return out

In [ ]:
class MLP(nn.Module):
  def __init__(self, d_model, d_ff):
    super().__init__()

    self.Sequential = nn.Sequential(
        nn.Linear(d_model, d_ff),
        nn.GELU(),
        nn.Linear(d_ff,d_model)
    )

  def forward(self,x):
    x = self.Sequential(x)
    return x

In [ ]:
class EncoderBlock(nn.Module):
  def __init__(self, d_model, num_heads, d_ff, drop_out=0.1):
    super().__init__()

    self.Attention = Attention(d_model, num_heads)
    self.MLP = MLP(d_model, d_ff)

    self.LayerNorm1 = nn.LayerNorm(d_model)
    self.LayerNorm2 = nn.LayerNorm(d_model)

    self.Dropout = nn.Dropout(drop_out)

  def forward(self,x):
    x_LN1 = self.LayerNorm1(x)
    x = self.Attention(x_LN1, x_LN1, x_LN1, mask = None) + x

    x_LN2 = self.LayerNorm2(x)
    x = self.MLP(x_LN2) + x

    return x

In [ ]:
class Encoder(nn.Module):
  def __init__(self, d_model, num_heads, d_ff, drop_out=0.1, L=12):
      super().__init__()

      self.L = L
      self.EncBlcockList = nn.ModuleList([
          EncoderBlock(d_model, num_heads, d_ff, drop_out)
          for i in range(L)
      ])

  def forward(self,x):
    for EncBlock in self.EncBlcockList:
      x = EncBlock(x)

    return x

In [ ]:
import math

class VisionTransformer(nn.Module):
  def __init__(self, pt_img_size, img_channel, patch_size, d_model, num_heads, d_ff, drop_out, L):
    super().__init__()

    self.pt_img_size = pt_img_size
    self.patch_size = patch_size
    self.d_model = d_model

    self.PatchEmbedding = PatchEmbedding(img_channel, patch_size, d_model)

    self.x_cls = nn.Parameter(torch.randn(1,1,d_model) * math.sqrt(2.0 / d_model)) # He init

    pt_N = pt_img_size**2 // patch_size**2
    self.pt_PositionalEncoding = nn.Parameter(torch.randn(1,pt_N+1, d_model) * math.sqrt(2.0 / d_model)) # He init

    self.Encoder = Encoder(d_model, num_heads, d_ff, drop_out, L)

  def pos_enc_interpolate(self, x, H, W):
    if H==W==self.pt_img_size:
      return self.pt_PositionalEncoding

    original_h = self.pt_img_size // self.patch_size
    original_w = self.pt_img_size // self.patch_size

    target_h = H // self.patch_size
    target_w = W // self.patch_size

    pos_cls = self.pt_PositionalEncoding[:,:1,:]
    pos_patch = self.pt_PositionalEncoding[:,1:,:]

    pos_patch = pos_patch.reshape(1,original_h,original_w,self.d_model).permute(0,3,1,2)
    pos_patch = F.interpolate(pos_patch, size=(target_h, target_w), mode="bicubic", align_corners=False)
    pos_patch = pos_patch.reshape(1,self.d_model,target_h*target_w).permute(0,2,1)

    return torch.cat([pos_cls,pos_patch], dim=1)


  def forward(self, x):
    B,C,H,W = x.shape

    x = self.PatchEmbedding(x) # (B,N,d_model)
    x = torch.cat((self.x_cls.expand(B,1,-1),x), dim=1) # (B,N+1,d_model)
    x = self.pos_enc_interpolate(x,H,W) + x

    x = self.Encoder(x)

    x = x[:,0,:]

    return x #(B,d_model)

In [ ]:
class ProjHead(nn.Module):
  def __init__(self, d_model, d_ff, num_classes):
    super().__init__()

    self.Sequential = nn.Sequential(
        nn.Linear(d_model, d_ff),
        nn.GELU(),
        nn.Linear(d_ff, d_ff),
        nn.GELU(),
        nn.Linear(d_ff, num_classes)
    )

  def forward(self,x):
    x = self.Sequential(x)
    return x

In [ ]:
class LinearClassifier(nn.Module):
    def __init__(self, d_model, num_classes):
        super().__init__()

        self.linear = nn.Linear(d_model,num_classes)

    def forward(self,x):
        return self.linear(x)

STL-10

In [ ]:
from torchvision.datasets import STL10
from torchvision.transforms import v2
from torch.utils.data import Dataset,DataLoader

pretrain_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
])

train_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32,scale=True),
    v2.RandomResizedCrop((84,84), scale=(0.45,1)),
    v2.RandomHorizontalFlip(),
    v2.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4)
])

eval_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32,scale=True),
    v2.Resize((84,84))
])

unlabeled_dataset = STL10(
    root = './data',
    split = 'unlabeled',
    transform= pretrain_transform,
    download = True
)


train_dataset = STL10(
    root = './data',
    split = 'train',
    transform= train_transform,
    download = True
)

test_dataset = STL10(
    root = './data',
    split = 'test',
    transform= eval_transform,
    download = True
)

unlabeled_dataloader = DataLoader(unlabeled_dataset, batch_size = 192, shuffle = True)
train_dataloader = DataLoader(train_dataset, batch_size = 192, shuffle = True)
test_dataloader = DataLoader(test_dataset, shuffle = False)

Mini ImageNet

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from datasets import load_dataset

class MiniImageNetDataset(Dataset):
    def __init__(self,split: str,transform=None,cache_dir: str | None = None):
        self.dataset = load_dataset("timm/mini-imagenet",split=split,cache_dir=cache_dir)
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        sample = self.dataset[index]

        image = sample["image"].convert("RGB")
        label = sample["label"]

        if self.transform is not None:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

train_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32,scale=True),
    v2.RandomResizedCrop((84,84), scale=(0.45,1)),
    v2.RandomHorizontalFlip(),
    v2.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4)
])

eval_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32,scale=True),
    v2.Resize((84,84))
])

train_dataset = MiniImageNetDataset(split="train",transform=train_transform)
test_dataset = MiniImageNetDataset(split="test",transform=eval_transform)

batch_size = 192
num_workers = 1

train_dataloader = DataLoader(
    train_dataset,
    batch_size=192,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=False
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=192,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=False
)

Utils

In [ ]:
from torchvision.transforms import v2

def augmentation(x, size = 84):
    if size==84:
      x = v2.RandomResizedCrop(size, scale=(0.35,1.0))(x)
    else:
      x = v2.RandomResizedCrop(size, scale=(0.05,0.35))(x)
    x = v2.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1)(x)
    x = v2.RandomHorizontalFlip(0.5)(x)

    return x

def forward_models(models, x):
  for model in models:
    x = model(x)
  return x

def update_teacher(teacher, student, m=0.996):
  for param_t, param_s in zip(teacher.parameters(), student.parameters()):
      param_t.data = param_t.data * m + param_s.data * (1.0 - m)

def dino_loss(s,t,C,S_temp,T_temp):
  s_log_prob = F.log_softmax(s/S_temp, dim=-1)
  t_prob = F.softmax((t-C)/T_temp, dim=-1)

  loss = - (t_prob * s_log_prob).sum(dim=-1).mean()
  return loss

def forward_features(model, dataloader, device):
  train_features = []
  train_labels = []

  for _, (x, label) in enumerate(dataloader):
    x, label = x.to(device), label.to(device)
    with torch.no_grad():
      logits = model(x)
    train_features.append(logits)
    train_labels.append(label)

  train_features = torch.cat(train_features, dim=0).to(device)
  train_labels = torch.cat(train_labels, dim=0).to(device)

  return train_features, train_labels

def knn_eval(model, train_dataloader, test_dataloader, num_classes, device):
  model.eval()

  train_features, train_labels = forward_features(model, train_dataloader, device) #(5000,d_model)
  train_features = F.normalize(train_features, dim=-1)

  current=0
  for _, (x, label) in enumerate(test_dataloader):
    x, label = x.to(device), label.to(device)

    target_features = model(x) #(d_model)
    target_features = F.normalize(target_features, dim=-1)

    cos_similarity = target_features @ train_features.T #(1,5000)

    knn_similarity, index = cos_similarity.topk(k=20, dim=-1, largest=True) #(20)
    knn_label = train_labels[index] #(1,20)

    weights = torch.exp(knn_similarity / 0.07)

    class_score = torch.zeros(1,num_classes,device = device)
    class_score.scatter_add_(1, knn_label.long(), weights)

    prediction = class_score.argmax(dim=-1)
    current += (prediction == label).sum().item()

  return current / len(test_dataloader.dataset)

DINO Pre-Training

In [ ]:
import copy

K = 150
C = torch.zeros(K).to(device)
S_temp = 0.1
T_temp = 0.04
T_momentum = 0.996
C_momentum = 0.9
epochs = 30
warmup_epochs=3

student = VisionTransformer(pt_img_size=84, img_channel=3, patch_size=6, d_model=192, num_heads=6, d_ff=768, drop_out=0.1, L=8).to(device) # global: 84, local: 36
teacher = copy.deepcopy(student).to(device)
teacher.requires_grad_(False)

student_projhead = ProjHead(d_model=192, d_ff=768, num_classes=K).to(device)
teacher_projhead = copy.deepcopy(student_projhead).to(device)
teacher_projhead.requires_grad_(False)

student.train()
student_projhead.train()
teacher.eval()
teacher_projhead.eval()

loss_fn = dino_loss
optimizer = torch.optim.AdamW(list(student.parameters())+list(student_projhead.parameters()), lr = 5e-4, weight_decay= 1e-3)

scheduler_linear = torch.optim.lr_scheduler.LinearLR(optimizer=optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
scheduler_cos = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs-warmup_epochs, eta_min=1e-5)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer=optimizer, schedulers=[scheduler_linear,scheduler_cos], milestones=[warmup_epochs])

for epoch in range(epochs):
  epoch_loss=0
  for _, (x, label) in enumerate(unlabeled_dataloader):

    xg1, xg2, x1, x2 = augmentation(x, size=84), augmentation(x, size=84), augmentation(x, size=36), augmentation(x, size=36)
    xg1, xg2, x1, x2 = xg1.to(device), xg2.to(device), x1.to(device), x2.to(device)

    sg1, sg2, s1, s2 = forward_models([student, student_projhead],xg1), forward_models([student, student_projhead],xg2), forward_models([student, student_projhead],x1), forward_models([student, student_projhead],x2)
    with torch.no_grad():
      tg1, tg2 = forward_models([teacher, teacher_projhead],xg1), forward_models([teacher, teacher_projhead],xg2)

    optimizer.zero_grad()
    loss = [loss_fn(sg2,tg1,C,S_temp,T_temp), loss_fn(s1,tg1,C,S_temp,T_temp), loss_fn(s2,tg1,C,S_temp,T_temp), loss_fn(sg1,tg2,C,S_temp,T_temp), loss_fn(s1,tg2,C,S_temp,T_temp), loss_fn(s2,tg2,C,S_temp,T_temp)]
    loss = sum(loss) / len(loss)
    loss.backward()
    optimizer.step()

    with torch.no_grad():
      update_teacher(teacher, student, T_momentum)
      update_teacher(teacher_projhead, student_projhead, T_momentum)

      C = C_momentum * C + (1.0 - C_momentum) * torch.mean(torch.cat([tg1,tg2], dim=0), dim=0)
      epoch_loss += loss.item()

  print(f"epoch {epoch+1} loss: {epoch_loss / len(unlabeled_dataloader)}")

  scheduler.step()

Model Path

In [ ]:
MODEL_PATH = ''

kNN Probe

In [ ]:
teacher = VisionTransformer(pt_img_size=84, img_channel=3, patch_size=6, d_model=192, num_heads=6, d_ff=768, drop_out=0.1, L=8)
teacher.load_state_dict(torch.load(MODEL_PATH, map_location=device))
teacher.to(device)

num_classes=100
with torch.no_grad():
    knn_acc = knn_eval(teacher, train_dataloader, test_dataloader, num_classes=num_classes, device=device)
    print(f"kNN Probe Accuracy: {knn_acc}")

Linear Probe

In [ ]:
teacher = VisionTransformer(pt_img_size=84, img_channel=3, patch_size=6, d_model=192, num_heads=6, d_ff=768, drop_out=0.1, L=8)
teacher.load_state_dict(torch.load(MODEL_PATH, map_location=device))
teacher.to(device)

num_classes=100
teacher_linear_classifier = LinearClassifier(d_model=192, num_classes=num_classes).to(device)

teacher.requires_grad_(False)
teacher.eval()
teacher_linear_classifier.train()

epochs = 40
warmup_epochs= 4

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(teacher_linear_classifier.parameters(), lr = 5e-4, weight_decay=0.0)

scheduler_linear = torch.optim.lr_scheduler.LinearLR(optimizer=optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
scheduler_cos = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs-warmup_epochs, eta_min=1e-5)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer=optimizer, schedulers=[scheduler_linear,scheduler_cos], milestones=[warmup_epochs])

for epoch in range(epochs):
  epoch_loss=0
  for _, (x, label) in enumerate(train_dataloader):
    x, label = x.to(device), label.to(device)

    logits = forward_models([teacher,teacher_linear_classifier],x)

    optimizer.zero_grad()
    loss = loss_fn(logits, label)
    loss.backward()
    optimizer.step()

    epoch_loss += loss.item()

  if (epoch+1)%10==0:
      print(f"epoch {epoch+1} loss: {epoch_loss / len(train_dataloader)}")
  scheduler.step()


In [ ]:
teacher.eval()
teacher_linear_classifier.eval()

current = 0
with torch.no_grad():
    for _, (x, label) in enumerate(test_dataloader):
        x, label = x.to(device), label.to(device)

        logits = forward_models([teacher,teacher_linear_classifier],x)

        current += (logits.argmax(dim=-1) == label).sum()
linear_acc = current / len(test_dataset)
print(f"Linear Probe Accuracy: {linear_acc}")

Fine Tune

In [ ]:
teacher = VisionTransformer(pt_img_size=84, img_channel=3, patch_size=6, d_model=192, num_heads=6, d_ff=768, drop_out=0.1, L=8)
teacher.load_state_dict(torch.load(MODEL_PATH, map_location=device))
teacher.to(device)

num_classes=100
teacher_linear_classifier = LinearClassifier(d_model=192, num_classes=num_classes).to(device)

teacher.train()
teacher_linear_classifier.train()

epochs = 40
warmup_epochs= 4

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(list(teacher.parameters())+list(teacher_linear_classifier.parameters()), lr = 5e-4, weight_decay= 1e-3)

scheduler_linear = torch.optim.lr_scheduler.LinearLR(optimizer=optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
scheduler_cos = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs-warmup_epochs, eta_min=1e-5)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer=optimizer, schedulers=[scheduler_linear,scheduler_cos], milestones=[warmup_epochs])

for epoch in range(epochs):
  epoch_loss=0
  for _, (x, label) in enumerate(train_dataloader):
    x, label = x.to(device), label.to(device)

    logits = forward_models([teacher,teacher_linear_classifier],x)

    optimizer.zero_grad()
    loss = loss_fn(logits, label)
    loss.backward()
    optimizer.step()

    epoch_loss += loss.item()

  if (epoch+1)%10==0:
      print(f"epoch {epoch+1} loss: {epoch_loss / len(train_dataloader)}")
  scheduler.step()


In [ ]:
teacher.eval()
teacher_linear_classifier.eval()

current = 0
with torch.no_grad():
    for _, (x, label) in enumerate(test_dataloader):
        x, label = x.to(device), label.to(device)

        logits = forward_models([teacher,teacher_linear_classifier],x)

        current += (logits.argmax(dim=-1) == label).sum()
fine_tuned_acc = current / len(test_dataset)
print(f"Fine-Tuned ViT Accuracy: {fine_tuned_acc}")

Random Initialized Supervised ViT

In [ ]:
rand_vit = VisionTransformer(pt_img_size=84, img_channel=3, patch_size=6, d_model=192, num_heads=6, d_ff=768, drop_out=0.1, L=8).to(device)

num_classes=100
rand_head = ProjHead(d_model=192, d_ff=784, num_classes=num_classes).to(device)

rand_vit.train()
rand_head.train()

epochs = 40
warmup_epochs = 4

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(list(rand_vit.parameters())+list(rand_head.parameters()), lr = 5e-4, weight_decay= 1e-3)

scheduler_linear = torch.optim.lr_scheduler.LinearLR(optimizer=optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
scheduler_cos = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs-warmup_epochs, eta_min=1e-5)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer=optimizer, schedulers=[scheduler_linear,scheduler_cos], milestones=[warmup_epochs])

for epoch in range(epochs):
  epoch_loss=0
  for _, (x, label) in enumerate(train_dataloader):
    x, label = x.to(device), label.to(device)

    logits = forward_models([rand_vit, rand_head],x)

    optimizer.zero_grad()
    loss = loss_fn(logits, label)
    loss.backward()
    optimizer.step()

    epoch_loss += loss.item()

  if (epoch+1)%10==0:
      print(f"epoch {epoch+1} loss: {epoch_loss / len(train_dataloader)}")
  scheduler.step()


In [ ]:
rand_vit.eval()
rand_head.eval()

current = 0
with torch.no_grad():
    for _, (x, label) in enumerate(test_dataloader):
        x, label = x.to(device), label.to(device)

        logits = forward_models([rand_vit,rand_head],x)

        current += (logits.argmax(dim=-1) == label).sum()
rand_acc = current / len(train_dataset)
print(f"Random Supervised ViT Accuracy: {rand_acc}")

Attention Map Evolving Visualization

In [ ]:
epoch=30

idx = torch.randint(0,5000,(1,)).item()
idx = 1154 #2143, 2767
image, label = test_dataset[idx]
img = image.unsqueeze(0).to(device)

model = VisionTransformer(pt_img_size=84, img_channel=3, patch_size=6, d_model=192, num_heads=6, d_ff=768, drop_out=0.1, L=8)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()

last_attention = model.Encoder.EncBlcockList[-1].Attention
last_attention.save_attention = True

with torch.no_grad():
  out = model(img)

attention_map = last_attention.attention_map[:,:,0,:].squeeze(0)
image_np = image.permute(1,2,0).cpu().numpy()

fig, axes = plt.subplots(3,3,figsize=(15,15),dpi=120)
axes = axes.flatten()

axes[0].axis("off")
axes[1].imshow(image_np)
axes[1].set_title(f"Original: STL-10 test index {idx}",fontsize=15)
axes[1].axis("off")
axes[2].axis("off")

for head in range(6):
  head_values, head_indices = attention_map[head][1:].sort(descending=True)

  selected_indices = []
  mass = 0.0
  for i in range(len(head_values)):
    mass += head_values[i].item()
    selected_indices.append(head_indices[i].item())
    if mass >= 0.6:
      break

  patch_mask = torch.zeros(14*14)
  patch_mask[torch.tensor(selected_indices)] = 1
  patch_mask = patch_mask.reshape(1,1,14,14)

  pixel_mask = F.interpolate(patch_mask,size=image.shape[-2:],mode="nearest")[0,0]
  mask_np = pixel_mask.cpu().numpy()

  axes[head+3].imshow(image_np)
  axes[head+3].imshow(mask_np,alpha=0.45,vmin=0,vmax=1)
  axes[head+3].set_title(f"Head {head+1} ({len(selected_indices)} patches)",fontsize=15)
  axes[head+3].axis("off")

plt.tight_layout()
plt.show()

Multi-Crop Ablation

In [ ]:
import copy

K = 150
C = torch.zeros(K).to(device)
S_temp = 0.1
T_temp = 0.04
T_momentum = 0.996
C_momentum = 0.9
epochs = 30
warmup_epochs=3

student = VisionTransformer(pt_img_size=84, img_channel=3, patch_size=6, d_model=192, num_heads=6, d_ff=768, drop_out=0.1, L=8).to(device) # global: 84, local: 36
teacher = copy.deepcopy(student).to(device)
teacher.requires_grad_(False)

student_projhead = ProjHead(d_model=192, d_ff=768, num_classes=K).to(device)
teacher_projhead = copy.deepcopy(student_projhead).to(device)
teacher_projhead.requires_grad_(False)

student.train()
student_projhead.train()
teacher.eval()
teacher_projhead.eval()

loss_fn = dino_loss
optimizer = torch.optim.AdamW(list(student.parameters())+list(student_projhead.parameters()), lr = 5e-4, weight_decay= 1e-3)

scheduler_linear = torch.optim.lr_scheduler.LinearLR(optimizer=optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
scheduler_cos = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs-warmup_epochs, eta_min=1e-5)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer=optimizer, schedulers=[scheduler_linear,scheduler_cos], milestones=[warmup_epochs])

for epoch in range(epochs):
  epoch_loss=0
  for _, (x, label) in enumerate(unlabeled_dataloader):

    xg1, xg2 = augmentation(x, size=84), augmentation(x, size=84)
    xg1, xg2 = xg1.to(device), xg2.to(device)

    sg1, sg2 = forward_models([student, student_projhead],xg1), forward_models([student, student_projhead],xg2)
    with torch.no_grad():
      tg1, tg2 = forward_models([teacher, teacher_projhead],xg1), forward_models([teacher, teacher_projhead],xg2)

    optimizer.zero_grad()
    loss = [loss_fn(sg2,tg1,C,S_temp,T_temp), loss_fn(sg1,tg2,C,S_temp,T_temp)]
    loss = sum(loss) / len(loss)
    loss.backward()
    optimizer.step()

    with torch.no_grad():
      update_teacher(teacher, student, T_momentum)
      update_teacher(teacher_projhead, student_projhead, T_momentum)

      C = C_momentum * C + (1.0 - C_momentum) * torch.mean(torch.cat([tg1,tg2], dim=0), dim=0)
      epoch_loss += loss.item()

  print(f"epoch {epoch+1} loss: {epoch_loss / len(unlabeled_dataloader)}")

  scheduler.step()


In [ ]:
teacher = VisionTransformer(pt_img_size=84, img_channel=3, patch_size=6, d_model=192, num_heads=6, d_ff=768, drop_out=0.1, L=8)
teacher.load_state_dict(torch.load(MODEL_PATH, map_location=device))
teacher.to(device)

with torch.no_grad():
    knn_acc = knn_eval(teacher, train_dataloader, test_dataloader, device)
    print(f"Global Only kNN Probe Accuracy: {knn_acc}")